# Economic Dispatch Walkthrough

Builds a small 3-bus system with a mix of quadratic- and piecewise-cost generators, a renewable, and a storage unit, then solves it and visualizes the result.

> Provenance note: reconstructed for this repo (see `PROJECT_BRIEF.md`) rather than transcribed from the original chat.

In [ ]:
from ed_model.data.schema import Bus, Line, Generator, CostSegment, Renewable, Storage, System
from ed_model.model.builder import build_ed_model
from ed_model.solve import solve_ed, recommended_solver
from ed_model.viz import plot_dispatch_stack, plot_lmp_heatmap, plot_storage_soc, plot_line_loading

## 1. Build the system

In [ ]:
system = System(
    buses=[
        Bus(name="A", is_reference=True),
        Bus(name="B"),
        Bus(name="C"),
    ],
    lines=[
        Line(name="L1", from_bus="A", to_bus="B", susceptance=10.0, limit=120),
        Line(name="L2", from_bus="B", to_bus="C", susceptance=10.0, limit=80),
    ],
    generators=[
        Generator(
            name="G1", bus="A", p_min=0, p_max=250,
            c2=0.01, c1=12, c0=0,
            ramp_up=80, ramp_down=80, p_initial=100,
        ),
        Generator(
            name="G2", bus="C", p_min=0, p_max=150,
            cost_type="piecewise",
            c0=50,
            segments=(
                CostSegment(width=75, marginal_cost=18),
                CostSegment(width=75, marginal_cost=24),
            ),
            ramp_up=60, ramp_down=60, p_initial=50,
        ),
    ],
    renewables=[
        Renewable(name="Wind1", bus="B", forecast=[40, 70]),
    ],
    storages=[
        Storage(
            name="Batt1", bus="B",
            energy_capacity=100, charge_limit=40, discharge_limit=40,
            charge_efficiency=0.95, discharge_efficiency=0.95, initial_soc=50,
        ),
    ],
    demand={"A": [120, 150], "B": [60, 90], "C": [70, 80]},
)
system

## 2. Build and solve

In [ ]:
model = build_ed_model(system)
solver_name = recommended_solver(system)
print(f"Using solver: {solver_name}")
result = solve_ed(model, solver_name=solver_name)
print(f"Total cost: ${result.total_cost:,.2f}")

## 3. Inspect dispatch and prices

In [ ]:
for t in range(1, system.n_periods + 1):
    print(f"Period {t}: dispatch={result.dispatch_for_period(t)}, LMPs={result.lmp_for_period(t)}")

## 4. Visualize

In [ ]:
plot_dispatch_stack(system, result)

In [ ]:
plot_lmp_heatmap(system, result)

In [ ]:
plot_storage_soc(system, result)

In [ ]:
plot_line_loading(system, result)